### LIBRARY IMPORTS

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.gb_classifier import GBClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)
X_test, y_test = processor.split_features_target(test)

y_train, y_valid, y_test = processor.transform_target(y_train, y_valid, y_test)

### LOGISTIC REGRESSION

In [ ]:
%%time

logistic = LogisticRegression(C=1.0, random_state=42)
logistic.fit(X_train, y_train)

logistic_valid_preds = logistic.predict(X_valid)
logistic_valid_probs = logistic.predict_proba(X_valid)

logistic_test_preds = logistic.predict(X_test)
logistic_test_probs = logistic.predict_proba(X_test)

print(f"Logistic validation log loss: {log_loss(y_valid, logistic_valid_probs):.4f}")
print(f"Logistic validation accuracy: {accuracy_score(y_valid, logistic_valid_preds):.4f}")
print('-' * 50)
print(f"Logistic test log loss: {log_loss(y_test, logistic_test_probs):.4f}")
print(f"Logistic test accuracy: {accuracy_score(y_test, logistic_test_preds):.4f}")
print('-' * 50)

Logistic validation log loss: 0.2713
Logistic validation accuracy: 0.8870
--------------------------------------------------
Logistic test log loss: 0.2747
Logistic test accuracy: 0.8848
--------------------------------------------------
CPU times: total: 688 ms
Wall time: 148 ms


### RANDOM FOREST

In [16]:
%%time

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

rf_valid_preds = rf.predict(X_valid)
rf_valid_probs = rf.predict_proba(X_valid)

rf_test_preds = rf.predict(X_test)
rf_test_probs = rf.predict_proba(X_test)

print(f"RF validation log loss: {log_loss(y_valid, rf_valid_probs):.4f}")
print(f"RF validation accuracy: {accuracy_score(y_valid, rf_valid_preds):.4f}")
print('-' * 50)
print(f"RF test log loss: {log_loss(y_test, rf_test_probs):.4f}")
print(f"RF test accuracy: {accuracy_score(y_test, rf_test_preds):.4f}")
print('-' * 50)

RF validation log loss: 0.2887
RF validation accuracy: 0.8817
--------------------------------------------------
RF test log loss: 0.2921
RF test accuracy: 0.8801
--------------------------------------------------
CPU times: total: 6.61 s
Wall time: 6.6 s


### NEURAL NETWORK

In [10]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            print(f"Epoch: {epoch + 1} | Validation log loss: {val_loss:.4f} | Validation accuracy {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        X_t = torch.from_numpy(X).to(torch.float32)
        X_t = X_t.to(self.device)

        self.eval()
        with torch.no_grad():
            predictions = self.forward(X_t)
        
        return torch.softmax(predictions, dim=1).cpu().numpy()

In [ ]:
%%time

mlp = MLP(epochs=100, learning_rate=0.0001, hidden_size=[64, 32], batch_size=256)
mlp.fit(X_train, y_train, X_valid, y_valid)

mlp_valid_preds = mlp.predict(X_valid)
mlp_valid_probs = mlp.predict_proba(X_valid)

mlp_test_preds = mlp.predict(X_test)
mlp_test_probs = mlp.predict_proba(X_test)

print('-' * 50)
print(f"MLP validation log loss: {log_loss(y_valid, mlp_valid_probs):.4f}")
print(f"MLP validation accuracy: {accuracy_score(y_valid, mlp_valid_preds):.4f}")
print('-' * 50)
print(f"MLP test log loss: {log_loss(y_test, mlp_test_probs):.4f}")
print(f"MLP test accuracy: {accuracy_score(y_test, mlp_test_preds):.4f}")
print('-' * 50)

Epoch: 1 | Validation log loss: 0.2901 | Validation accuracy 0.8845
Epoch: 2 | Validation log loss: 0.2730 | Validation accuracy 0.8864
Epoch: 3 | Validation log loss: 0.2722 | Validation accuracy 0.8865
Epoch: 4 | Validation log loss: 0.2719 | Validation accuracy 0.8863
Epoch: 5 | Validation log loss: 0.2718 | Validation accuracy 0.8862
Epoch: 6 | Validation log loss: 0.2715 | Validation accuracy 0.8865
Epoch: 7 | Validation log loss: 0.2711 | Validation accuracy 0.8864
Epoch: 8 | Validation log loss: 0.2712 | Validation accuracy 0.8860
Epoch: 9 | Validation log loss: 0.2710 | Validation accuracy 0.8865
Epoch: 10 | Validation log loss: 0.2711 | Validation accuracy 0.8865
Epoch: 11 | Validation log loss: 0.2711 | Validation accuracy 0.8870
Epoch: 12 | Validation log loss: 0.2716 | Validation accuracy 0.8870
Epoch: 13 | Validation log loss: 0.2710 | Validation accuracy 0.8864
Epoch: 14 | Validation log loss: 0.2712 | Validation accuracy 0.8873
Epoch: 15 | Validation log loss: 0.2708 | V

### GRADIENT BOOSTING (DECISION TREES)

In [6]:
gb = GBClassifier(**gradient_boosting_config, weak_learner_config=weak_learner_config)
gb.load_model("models/heart/2026_04_28_19_09/model.joblib")

gb_valid_preds = gb.predict(X_valid)
gb_valid_probs = gb.predict_proba(X_valid)

gb_test_preds = gb.predict(X_test)
gb_test_probs = gb.predict_proba(X_test)

print('-' * 50)
print(f"GB (DTs) validation log loss: {log_loss(y_valid, gb_valid_probs):.4f}")
print(f"GB (DTs) validation accuracy: {accuracy_score(y_valid, gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (DTs) test log loss: {log_loss(y_test, gb_test_probs):.4f}")
print(f"GB (DTs) test accuracy: {accuracy_score(y_test, gb_test_preds):.4f}")

2026-04-29 20:41:36,106 - INFO - Model loaded from models/heart/2026_04_28_19_09/model.joblib


--------------------------------------------------
GB (DTs) validation log loss: 0.2668
GB (DTs) validation accuracy: 0.8902
--------------------------------------------------
GB (DTs) test log loss: 0.2717
GB (DTs) test accuracy: 0.8872


### GRADIENT BOOSTING (NEURAL NETWORKS)

In [5]:
nn_gb = GBClassifier(**gradient_boosting_config, weak_learner_config=weak_learner_config)
nn_gb.load_model("models/heart/2026_04_28_22_41/model.joblib")

nn_gb_valid_preds = nn_gb.predict(X_valid)
nn_gb_valid_probs = nn_gb.predict_proba(X_valid)

nn_gb_test_preds = nn_gb.predict(X_test)
nn_gb_test_probs = nn_gb.predict_proba(X_test)

print('-' * 50)
print(f"GB (NNs) validation log loss: {log_loss(y_valid, nn_gb_valid_probs):.4f}")
print(f"GB (NNs) validation accuracy: {accuracy_score(y_valid, nn_gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (NNs) test log loss: {log_loss(y_test, nn_gb_test_probs):.4f}")
print(f"GB (NNs) test accuracy: {accuracy_score(y_test, nn_gb_test_preds):.4f}")

2026-04-29 20:40:43,362 - INFO - Model loaded from models/heart/2026_04_28_22_41/model.joblib


--------------------------------------------------
GB (NNs) validation log loss: 0.2717
GB (NNs) validation accuracy: 0.8873
--------------------------------------------------
GB (NNs) test log loss: 0.2758
GB (NNs) test accuracy: 0.8856
